# 02 — Training Dynamics

KAMUI's training loop is a visible for-loop: forward, backward, clip,
scheduled LR, step. This notebook trains a tiny model and plots the two
curves every practitioner should recognise: the loss and the LR schedule.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
# Train a tiny model on a synthetic corpus (~30s on CPU).
# The corpus is a seeded word-salad: repetitive enough to learn, varied enough
# that BPE cannot collapse it into a handful of giant tokens.
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
WORDS = ["the", "cat", "dog", "sat", "ran", "on", "to", "mat", "log", "sun"]
CORPUS = " ".join(rng.choice(WORDS) for _ in range(4000))

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=300, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(150)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

steps = [r["step"] for r in records]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(steps, [r["train_loss"] for r in records]); axes[0].set_title("train loss")
axes[1].plot(steps, [r["lr"] for r in records]); axes[1].set_title("LR (warmup + cosine)")
axes[2].plot(steps, [r["grad_norm"] for r in records]); axes[2].set_title("grad norm (post-clip)")
for ax in axes: ax.set_xlabel("step")
fig.tight_layout()
fig

In [ ]:
# The trained model now continues our toy corpus sensibly.
print(kamui.generate(model, tokenizer, "the cat ", max_new_tokens=15, strategy="greedy"))